In [ ]:
# Demo 7: Claim Quality Test (LLM parameter: temperature)

# Use to get hugging face models
!pip install transformers accelerate huggingface_hub
# Use python library to get youtube transcript
!pip install youtube-transcript-api

In [ ]:
# Libraries
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from transformers import AutoTokenizer

# Integrate youtube_api.py code:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd

# Load tokenizer with left padding for Qwen
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", padding_side ='left')


In [ ]:

# Integrate youtube_api.py code:

load_dotenv()

# api_key= os.getenv("API_KEY")

yta = YouTubeTranscriptApi()

"""
def get_transcript(video_id):
    try:
        transcript = yta.fetch(video_id)
        transcript_text = " ".join(entry.text for entry in transcript)
        return transcript_text
    except Exception:
        return None
"""

youtube = build('youtube', 'v3', developerKey=api_key)

request = youtube.videos().list(
    part='localizations,statistics,topicDetails',
    chart='mostPopular',
    regionCode='us',
    videoCategoryId='20',
    maxResults='50'
)

response = request.execute()

videos_data = []

for item in response["items"]:
    video_id = item.get("id")

    videos_data.append({"Video ID": video_id})

df = pd.DataFrame(videos_data)
# df["Transcript"] = df["Video ID"].apply(get_transcript)

In [ ]:
# Load Qwen model (small free version)
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    tokenizer = tokenizer,
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=200 # increase to prevent incomplete output
)

In [5]:
# Clean raw transcript

def clean_transcript(transcript):
  cleaned = [] # list for cleaned snippets

  # Loop to get text (clean) & get start time
  for snippet in transcript.snippets:
    text = snippet.text # get text from snippet
    start_time = snippet.start # get start time from snippet

    while "[" in text:
      start = text.index("[")
      end = text.index("]") + 1
      caption = text[start:end]
      text = text.replace(caption, "")

    cleaned.append({"text": text, "start": start_time}) # add snippet's text and start time to cleaned list

  # Test print cleaned list
  # print(f"Transcript Text:\n{cleaned}")

  return cleaned

In [6]:
# Chunk cleaned transcript

def chunk_transcript(cleaned):
  chunk_size = 300 # chunk size based on time (sec)
  chunks = [] # list for chunks
  chunk_snippets = [] # list to build each chunk
  curr_chunk_time = 0

  # Loop through all snippets in cleaned[]
  for snippet in cleaned:
      text = snippet["text"] # get snippet text
      chunk_snippets.append(text) # add text to chunk_snippets list

      # Check if reach chunk_size (approx.)
      if snippet["start"] - curr_chunk_time >= chunk_size:
        # print(f"Chunk time: {snippet["start"] - curr_chunk_time} sec") # Test print chunk start time
        chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
        chunks.append(chunk) # add chunk to list of chunks

        chunk_snippets = [] # reset chunk_snippets list
        curr_chunk_time = snippet["start"] # reset time to current start time

  # If leftover snippets, create and add last chunk
  if chunk_snippets:
    chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
    chunks.append(chunk) # add chunk to list of chunks

  # Test print chunks list
  # print(f"Transcript Text:\n{chunks}")

  return chunks


In [7]:
# Claim extraction step (chunk batching)

# Function to extract claims
def claim_extraction(chunks):
  message_list = []
  # Create message more each chunk
  for chunk in chunks:
    messages = [
      {"role": "system", "content": "You are a helpful assistant that extracts factual claims from text."},
      # {"role": "user", "content": f"Extract all factual claims as bullet points.\n\nText: {chunk}"}
      {"role": "user", "content": f"Extract one concise high-level claim that summarizes the main point.\n\nText: {chunk}"}
    ]
    message_list.append(messages) # add message to list

  results = []
  # Send messages to Qwen model (process in batches)
  # Test temperature parameter (default 0.7)
  for output in pipe(message_list, batch_size=5, max_length=None, temperature=0.2):
    # Get claim from Qwen response
    result = output[0]["generated_text"][-1]["content"]
    results.append(result) # add qwen output to list
  return results



In [ ]:
# Test extract claim from multiple videos

# json library
import json

# 50 video ids from df into a list test:
video_ids = df["Video ID"].tolist()[:20] # test first 20
# video_ids = df["Video ID"].tolist() # test all 50

video_claims = []
# Loop through each video id in list
for video_id in video_ids:
  yta = YouTubeTranscriptApi()
  try:
    transcript = yta.fetch(video_id) # get transcript

    # Filter videos based on time (keep only <=30 minutes)
    video_time = transcript.snippets[-1].start + transcript.snippets[-1].duration
    if video_time <= 1800:

      chunks = chunk_transcript(clean_transcript(transcript)) # clean/chunk transcript

      # Extract claims from chunks list
      claims_list = claim_extraction(chunks)

      video_claims.append({"video_id": video_id, "chunks": chunks, "claims": claims_list})
      """
      # Print claims
      for claims in claims_list:
        print(f"Claims Extracted:\n{claims}\n")
      """
    """
    else:
      print(f"Video ID: {video_id} is too long")
    """

  except Exception as e: # error for no transcript
    # print(f"No transcript for video ID: {video_id}")
    print(f"Error for {video_id}: {type(e).__name__}: {e}")

with open('video_claims.json', 'w') as file:
  json.dump(video_claims, file, indent=4)




Test LLM paramter: temperature

Temp: 0.7 (default)

    {
        "video_id": "tbXDaBQDJf8",
        "claims": [
            "The reviewer discusses their experience with Pok\u00e9mon Champions, highlighting both positive aspects such as the quality of gameplay during battles, particularly in the Champion Stadium, and criticisms regarding the user interface, technical performance, lack of social features, and customization options.",
            "The game allows players to build custom teams of Pok\u00e9mon using various types of movesets and battle strategies, focusing primarily on fire-type teams with Mega Charizards due to its unique features such as daily missions, battle pass rewards, and customizable character appearances.",
            "The main point seems to be about discussing the benefits and mechanics of Pok\u00e9mon Home, a service offering access to various Pok\u00e9mon games without purchasing individual titles.",
            "Champions offers a casual Pok\u00e9mon battle experience featuring unique abilities such as Piercing Drill and Unseen Fist, with potential for future updates based on player feedback.",
            "The text discusses changes and updates to a show or content series, mentioning new elements such as \"megas\" and a \"new season.\""
        ]
    },
- some misinterpretations of main idea

Temp: 0.2

    {
        "video_id": "tbXDaBQDJf8",
        "claims": [
            "The reviewer discusses their initial experience with Pokemon Champions, noting positive aspects such as the visual quality during battles and the lack of need to manually adjust frame rates, while also pointing out issues like poor user interface design, technical lag, and limited customization options.",
            "The game allows players to create custom teams of Pok\u00e9mon based on various criteria such as ability, stat, and move set, offering both competitive and casual gameplay experiences.",
            "The main point seems to be about the economics and accessibility aspects of Pok\u00e9mon Home compared to traditional Pok\u00e9mon games, focusing on the low-cost nature of acquiring Pok\u00e9mon through various methods (Terror Raid Battles, Dynamax battles, Legends EA items) versus the higher costs associated with purchasing starter packs or DLC content.",
            "Champions introduces new abilities and mechanics that change how certain Pok\u00e9mon interact with each other, potentially altering their effectiveness based on their Mega Evolution status.",
            "The content of the video has changed significantly since its previous release, with new characters and elements introduced."
        ]
    },

- less misinterpretations of main idea
- less random/ more accurate to transcript

 Temp: 1.0

    {
        "video_id": "tbXDaBQDJf8",
        "claims": [
            "The reviewer provides an initial assessment of Pok\u00e9mon Champions, focusing on its gameplay mechanics, visual quality during battles, issues related to frame rate, technical difficulties, and the lack of integration between mobile and console versions.",
            "The game allows players to build customizable avatars based on their Pok\u00e9mon, but lacks features such as battling competitively offline or obtaining additional items before its launch in four months.",
            "The main point appears to be about the value proposition and monetization of Pok\u00e9mon Home within the context of Pok\u00e9mon games and related content.",
            "Champions is a casual Pok\u00e9mon battle experience offering competitive battles for existing Pok\u00e9mon users, featuring new abilities like Piercing Drill and Unseen Fist. It faces criticism due to its low-quality visuals and slow performance on Nintendo Switch, despite having some positive aspects such as improved Pok\u00e9mon models and movesetting adjustments.",
            "The show has undergone significant changes since its last appearance."
        ]
    }

- more misinterpretations from main idea
- most inaccurate (mix up content from transcript)
- less concise